# Traffic Sign Classification
هنقارن CNN بسيطة مع ResNet-18 مدرّبة مسبقًا. كل صورة فيها إشارة واحدة، والمطلوب نحدد فئتها من 43 فئة.

## تشغيل التجربة
في Colab اختار GPU وارفع `train.py` في Files. في Kaggle اختار GPU وأضف `train.py` كملف داخل Dataset خاص. فعّل الإنترنت لتنزيل GTSRB والأوزان. لو الملف في مسار مختلف عدّل `script` تحت.

In [ ]:
from pathlib import Path
import subprocess
import sys

script = Path("train.py")
if not script.is_file() and Path("/kaggle/input").is_dir():
    candidates = list(Path("/kaggle/input").rglob("train.py"))
    if len(candidates) == 1:
        script = candidates[0]
if not script.is_file():
    raise FileNotFoundError("Upload train.py or set script to its full path.")

import torch
print(torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU runtime")

## من الصورة للتوقع
الصورة بتبدأ بـ3 قنوات و96×96. أول MaxPool بيصغّرها لـ48×48 والتاني لـ24×24. AdaptiveAvgPool بيحوّل خرائط الخصائص لـ4×4 قبل طبقة التوقع. هنستخدم Macro-F1 عشان كل فئة يبقى ليها نفس الوزن في التقييم.

In [ ]:
from uuid import uuid4

output = Path("runs") / f"experiment-{uuid4().hex[:8]}"
subprocess.run(
    [sys.executable, str(script), "--epochs", "5", "--batch-size", "64", "--output", str(output)],
    check=True,
)

## قراءة النتيجة
قارن الـtest Macro-F1 للموديلين وافتح ملفات الـconfusion matrix. بص على أول زوج إشارات حصل بينهم لخبطة. بعدين نزّل مجلد النتائج لو عايز تحتفظ بيه.

In [ ]:
import json

report = json.loads((output / "metrics.json").read_text())
for name, result in report["models"].items():
    print(name, result["test"])
    print("Most confused classes:", result["top_confusions"][:1])
print(output.resolve())